In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

***EDA***

In [ ]:
import pandas as pd

train_path = "/kaggle/input/competitions/nppe-dlp-2026-term-1/train.csv"
test_path = "/kaggle/input/competitions/nppe-dlp-2026-term-1/test.csv"
sample_path = "/kaggle/input/competitions/nppe-dlp-2026-term-1/sample_submission.csv"

train = pd.read_csv(train_path)
test = pd.read_csv(test_path)
sample = pd.read_csv(sample_path)

print("Train Head")
print(train.head())

print("TEST DATA")
display(test.head())

print("Train shape:", train.shape)
print("Test shape:", test.shape)

print("Data types")
print(train.dtypes)

print("missing values in train data")
print(train.isnull().sum())

print("NUMERICAL STATISTICS")
display(train.describe())

print("Sample Submission Format")
display(sample.head())

print("Submission columns:")
print(sample.columns)

In [1]:
!pip uninstall -y torchvision torchaudio
!pip install -q transformers==4.50.0 peft==0.14.0 trl==0.12.0 accelerate==0.34.0 datasets tokenizers==0.21.0
!pip install -q -U bitsandbytes

In [2]:
import os
import pandas as pd
import numpy as np
import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
)
from peft import LoraConfig, get_peft_model, TaskType, PeftModel
from trl import SFTTrainer
import warnings
warnings.filterwarnings('ignore')

print('PyTorch version:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

2026-03-07 01:05:09.536907: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1772845509.562015     894 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1772845509.569899     894 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1772845509.589978     894 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772845509.590011     894 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772845509.590013     894 computation_placer.cc:177] computation placer alr

PyTorch version: 2.3.1+cu121
CUDA available: True
GPU: Tesla P100-PCIE-16GB


In [3]:
BASE_PATH = '/kaggle/input/competitions/nppe-dlp-2026-term-1/'

train_df = pd.read_csv(BASE_PATH + 'train.csv')
test_df  = pd.read_csv(BASE_PATH + 'test.csv')

print('Train shape:', train_df.shape)
print('Test shape:', test_df.shape)
print(train_df['label'].value_counts())
print()
print('Language distribution:')
print(train_df['language'].value_counts())
print()
print(train_df.groupby(['language', 'label']).size().unstack(fill_value=0))
train_df.head()

Train shape: (900, 4)
Test shape: (100, 3)
label
Positive    456
Negative    444
Name: count, dtype: int64

Language distribution:
language
ta    76
hi    74
or    72
pa    72
as    71
bd    71
gu    69
ml    68
mr    67
kn    66
ur    66
bn    65
te    63
Name: count, dtype: int64

label     Negative  Positive
language                    
as              35        36
bd              35        36
bn              31        34
gu              34        35
hi              35        39
kn              33        33
ml              33        35
mr              36        31
or              35        37
pa              38        34
ta              37        39
te              29        34
ur              33        33


,ID,sentence,label,language
0,82,ਇਹ ਫਿਲਮ ਇੱਕ ਬੇਹਤਰੀਨ ਕਹਾਣੀ ਸੁਣਾਉਣ ਦਾ ਸਭ ਤੋਂ ਵਧੀ...,Negative,pa
1,618,"ਇੱਕ ਸਮਗਰੀ ਦੇ ਰੂਪ ਵਿੱਚ, ਕੋਟਿੰਗ ਤੋਂ ਬਿਨਾਂ, ਸਿਰਫ ...",Negative,pa
2,812,"ബ്രിസിലുകൾ കട്ടിയുള്ള പ്ലാസ്റ്റിക് ആണ്, അതിനാൽ...",Negative,ml
3,304,এটি বিআইএস প্রত্যয়িত এবং নিরাপদ বলে দাবি করা হয়।,Positive,bn
4,295,6 mAh બેટરી અને લાંબા સમય સુધી ચાલતી નથી.,Negative,gu


In [4]:
lang_map = {
    'as': 'Assamese', 'bd': 'Bodo',    'bn': 'Bengali',
    'gu': 'Gujarati', 'hi': 'Hindi',   'kn': 'Kannada',
    'ml': 'Malayalam','mr': 'Marathi', 'or': 'Odia',
    'pa': 'Punjabi',  'ta': 'Tamil',   'te': 'Telugu',
    'ur': 'Urdu'
}

def build_prompt_train(row):
    lang = lang_map.get(row['language'], row['language'])
    return (
        f"<start_of_turn>user\n"
        f"Classify the sentiment of the following {lang} text as exactly one word: Positive or Negative.\n"
        f"Text: {row['sentence']}\n"
        f"<end_of_turn>\n"
        f"<start_of_turn>model\n"
        f"{row['label']}<end_of_turn>"
    )

def build_prompt_test(row):
    lang = lang_map.get(row['language'], row['language'])
    return (
        f"<start_of_turn>user\n"
        f"Classify the sentiment of the following {lang} text as exactly one word: Positive or Negative.\n"
        f"Text: {row['sentence']}\n"
        f"<end_of_turn>\n"
        f"<start_of_turn>model\n"
    )

train_df['text'] = train_df.apply(build_prompt_train, axis=1)
print(train_df['text'].iloc[0])

<start_of_turn>user
Classify the sentiment of the following Punjabi text as exactly one word: Positive or Negative.
Text: ਇਹ ਫਿਲਮ ਇੱਕ ਬੇਹਤਰੀਨ ਕਹਾਣੀ ਸੁਣਾਉਣ ਦਾ ਸਭ ਤੋਂ ਵਧੀਆ ਉਦਾਹਰਣ ਹੈ, ਪਰ ਇਹ ਸਭ ਨਕਾਰਾਤਮਕ ਅਰਥਾਂ ਵਿੱਚ ਕੀਤਾ ਗਿਆ ਹੈ। ਤੁਹਾਨੂੰ ਹਰ ਦ੍ਰਿਸ਼ ਨੂੰ ਸਮਝਣ ਲਈ ਸਮਾਂ ਕੱਢਣਾ ਹੋਵੇਗਾ!
<end_of_turn>
<start_of_turn>model
Negative<end_of_turn>


In [5]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")
login(token=hf_token)
print("Logged in to HuggingFace.")

Logged in to HuggingFace.


In [6]:
MODEL_ID = 'google/gemma-3-1b-it'

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map='auto',
    torch_dtype=torch.bfloat16,
)
model.config.use_cache = False
print('Model loaded.')

Model loaded.


In [7]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'],
    lora_dropout=0.05,
    bias='none',
    task_type=TaskType.CAUSAL_LM,
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 2,981,888 || all params: 1,002,867,840 || trainable%: 0.2973


In [8]:
hf_dataset = Dataset.from_pandas(train_df[['text']])
print(hf_dataset)

Dataset({
    features: ['text'],
    num_rows: 900
})


In [9]:
training_args = TrainingArguments(
    output_dir='/kaggle/working/gemma-sentiment',
    num_train_epochs=4,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    warmup_steps=20,
    learning_rate=2e-4,
    fp16=False,
    bf16=True,
    logging_steps=20,
    save_strategy='epoch',
    optim='paged_adamw_8bit',
    lr_scheduler_type='cosine',
    report_to='none',
    seed=42,
)

trainer = SFTTrainer(
    model=model,
    train_dataset=hf_dataset,
    args=training_args,
    dataset_text_field='text',
    max_seq_length=256,
    tokenizer=tokenizer,
    packing=False,
)

trainer.train()

Map:   0%|          | 0/900 [00:00<?, ? examples/s]

No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.
It is strongly recommended to train Gemma3 models with the `eager` attention implementation instead of `sdpa`. Use `eager` with `AutoModelForCausalLM.from_pretrained('<path-to-checkpoint>', attn_implementation='eager')`.


Step,Training Loss
20,5.560100
40,3.338400
60,2.955100
80,2.656800
100,2.506500
120,2.411100
140,2.257600
160,2.358700
180,2.237600
200,2.245200


TrainOutput(global_step=224, training_loss=2.7919426219803944, metrics={'train_runtime': 484.5184, 'train_samples_per_second': 7.43, 'train_steps_per_second': 0.462, 'total_flos': 1759164410354688.0, 'train_loss': 2.7919426219803944, 'epoch': 3.942222222222222})

In [10]:
model.save_pretrained('/kaggle/working/gemma-sentiment-lora')
tokenizer.save_pretrained('/kaggle/working/gemma-sentiment-lora')
print('Adapter saved.')

Adapter saved.


In [11]:
model.eval()

def predict_sentiment(row, model, tokenizer, max_new_tokens=8):
    prompt = build_prompt_test(row)
    inputs = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=256).to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=1.0,
            pad_token_id=tokenizer.eos_token_id,
        )
    
    generated = outputs[0][inputs['input_ids'].shape[1]:]
    decoded = tokenizer.decode(generated, skip_special_tokens=True).strip()
    return decoded

def clean_prediction(raw):
    raw_lower = raw.lower().strip()
    if 'negative' in raw_lower:
        return 0
    elif 'positive' in raw_lower:
        return 1
    else:
        print(f'  [WARN] Unexpected output: "{raw}" — defaulting to 1')
        return 1

print('Running inference on test set...')
predictions = []

for i, row in test_df.iterrows():
    raw = predict_sentiment(row, model, tokenizer)
    label = clean_prediction(raw)
    predictions.append(label)
    if (i % 10) == 0:
        print(f'  {i+1}/{len(test_df)} — "{raw}" -> {label}')

print('Done.')

Running inference on test set...
  [WARN] Unexpected output: "." — defaulting to 1
  1/100 — "." -> 1
  11/100 — "Negative" -> 0
  21/100 — "Negative" -> 0
  31/100 — "Positive" -> 1
  41/100 — "Positive" -> 1
  51/100 — "Positive" -> 1
  61/100 — "Positive" -> 1
  71/100 — "Positive" -> 1
  81/100 — "Positive" -> 1
  91/100 — "Negative" -> 0
Done.


In [ ]:
submission = pd.DataFrame({
    'ID': test_df['ID'].values,
    'label': predictions
})

print(submission.head(10))
print('Distribution:', submission['label'].value_counts().to_dict())

submission.to_csv('/kaggle/working/submission.csv', index=False)

check = pd.read_csv('/kaggle/working/submission.csv')
assert len(check) == 100
assert set(check['label'].unique()).issubset({0, 1}), f"Bad labels: {check['label'].unique()}"
print('All checks passed. Ready to submit!')

In [12]:
from sklearn.metrics import f1_score, classification_report

val_df = train_df.sample(n=150, random_state=42).reset_index(drop=True)

val_preds = []
for _, row in val_df.iterrows():
    raw = predict_sentiment(row, model, tokenizer)
    val_preds.append(clean_prediction(raw))

# Convert true labels to match numeric format
true_labels = val_df['label'].map({'Positive': 1, 'Negative': 0}).tolist()

f1 = f1_score(true_labels, val_preds, average='macro')
print(f'Current model Validation F1: {f1:.4f}')
print()
print(classification_report(true_labels, val_preds, target_names=['Negative', 'Positive']))

Current model Validation F1: 0.9399

              precision    recall  f1-score   support

    Negative       0.95      0.94      0.94        78
    Positive       0.93      0.94      0.94        72

    accuracy                           0.94       150
   macro avg       0.94      0.94      0.94       150
weighted avg       0.94      0.94      0.94       150

